# Advanced Problems: Delegating Iterators

This notebook expands the topic of **delegating iterators** with advanced examples, problems, and complete solutions.

Delegating iteration means implementing `__iter__` by returning an iterator from an existing iterable, usually with `return iter(self._data)`.

The main idea:

```python
class MyContainer:
    def __iter__(self):
        return iter(self._some_existing_iterable)
```

Best-practice goals in this notebook:

- Return a **fresh iterator** each time `__iter__` is called.
- Keep internal data private while still allowing iteration.
- Know when to delegate to a list, tuple, dictionary view, generator, or iterator factory.
- Avoid iterator exhaustion bugs.
- Design multiple useful iteration views without exposing internals.
- Test custom iterables with `list(...)`, `for`, `sorted(...)`, comprehensions, `zip(...)`, and repeated passes.


## Setup

We will use only the Python standard library.


In [1]:
from __future__ import annotations

from collections import Counter, OrderedDict, namedtuple
from dataclasses import dataclass
from itertools import chain, islice
from typing import Callable, Iterable, Iterator, Sequence
import re


## Mini Recap: The Delegation Pattern

A class becomes iterable when it implements `__iter__`.

If the class already stores data in an iterable object such as a list, tuple, dictionary, set, or range, you often do **not** need to write a separate iterator class.

Instead, delegate:

```python
def __iter__(self):
    return iter(self._items)
```

The object returned by `iter(self._items)` is the actual iterator.


In [2]:
Person = namedtuple("Person", "first last")

class PersonNames:
    def __init__(self, persons: Iterable[Person]):
        self._names = tuple(
            f"{person.first.capitalize()} {person.last.capitalize()}"
            for person in persons
        )

    def __iter__(self) -> Iterator[str]:
        # Delegates iteration to the tuple iterator.
        return iter(self._names)

persons = [
    Person("michaeL", "paLin"),
    Person("eric", "idLe"),
    Person("john", "cLeese"),
]

names = PersonNames(persons)

print(list(names))
print(sorted(names, key=lambda full_name: full_name.split()[-1]))

# A good iterable can be traversed more than once.
assert list(names) == ["Michael Palin", "Eric Idle", "John Cleese"]
assert list(names) == ["Michael Palin", "Eric Idle", "John Cleese"]


['Michael Palin', 'Eric Idle', 'John Cleese']
['John Cleese', 'Eric Idle', 'Michael Palin']


---

# Problem 1: Immutable Name Directory

Create a `NameDirectory` class.

Requirements:

1. Accept an iterable of `Person(first, last)` objects.
2. Normalize each full name into title case.
3. Store the normalized names internally as an immutable tuple.
4. Delegate iteration to the internal tuple.
5. Support `len(directory)`.
6. Support membership checks such as `'John Cleese' in directory`.
7. Provide a `first_names()` method that returns an iterator over first names.
8. Make sure repeated iteration works.


In [3]:
# Problem 1 solution

class NameDirectory:
    def __init__(self, persons: Iterable[Person]):
        self._names = tuple(
            f"{person.first.strip().title()} {person.last.strip().title()}"
            for person in persons
        )

    def __iter__(self) -> Iterator[str]:
        # Delegating to tuple gives a fresh tuple_iterator each time.
        return iter(self._names)

    def __len__(self) -> int:
        return len(self._names)

    def __contains__(self, name: object) -> bool:
        return name in self._names

    def first_names(self) -> Iterator[str]:
        # This method delegates to a generator expression.
        return (full_name.split(maxsplit=1)[0] for full_name in self._names)

    def last_names(self) -> Iterator[str]:
        return (full_name.split(maxsplit=1)[1] for full_name in self._names)

    def as_tuple(self) -> tuple[str, ...]:
        # Safe read-only representation.
        return self._names

people = [
    Person("  graham", "chapman  "),
    Person("john", "cleese"),
    Person("terry", "gilliam"),
    Person("eric", "idle"),
]

directory = NameDirectory(people)

print(list(directory))
print(list(directory.first_names()))
print(list(directory.last_names()))

assert len(directory) == 4
assert "John Cleese" in directory
assert "john cleese" not in directory
assert list(directory) == list(directory)  # repeated iteration works
assert directory.as_tuple() == (
    "Graham Chapman",
    "John Cleese",
    "Terry Gilliam",
    "Eric Idle",
)


['Graham Chapman', 'John Cleese', 'Terry Gilliam', 'Eric Idle']
['Graham', 'John', 'Terry', 'Eric']
['Chapman', 'Cleese', 'Gilliam', 'Idle']


### Why this is a good design

`NameDirectory` stores a tuple, not a list. That makes accidental external mutation harder.

`__iter__` returns `iter(self._names)`, so every loop receives a fresh iterator.

This is better than returning `self`, because `NameDirectory` is an iterable container, not an iterator.


---

# Problem 2: Multiple Delegated Views Over a Roster

Create a `Roster` class for players.

Requirements:

1. Store `Player(first, last, number, position)` objects.
2. Iterating over `Roster` should yield the players in insertion order.
3. Add `names()` that iterates over full names.
4. Add `numbers()` that iterates over jersey numbers.
5. Add `by_position(position)` that lazily yields players matching a position.
6. Add `sorted_by_last_name()` that returns an iterator over players sorted by last name.
7. Add tests proving every view works.


In [4]:
# Problem 2 solution

@dataclass(frozen=True)
class Player:
    first: str
    last: str
    number: int
    position: str

class Roster:
    def __init__(self, players: Iterable[Player]):
        self._players = tuple(players)

    def __iter__(self) -> Iterator[Player]:
        # Main iteration delegates to the tuple of players.
        return iter(self._players)

    def __len__(self) -> int:
        return len(self._players)

    def names(self) -> Iterator[str]:
        return (f"{player.first} {player.last}" for player in self._players)

    def numbers(self) -> Iterator[int]:
        return (player.number for player in self._players)

    def by_position(self, position: str) -> Iterator[Player]:
        normalized = position.strip().upper()
        return (
            player
            for player in self._players
            if player.position.upper() == normalized
        )

    def sorted_by_last_name(self) -> Iterator[Player]:
        # sorted(...) returns a list; iter(...) delegates to that list's iterator.
        return iter(sorted(self._players, key=lambda player: (player.last, player.first)))

    def position_counts(self) -> Counter[str]:
        return Counter(player.position.upper() for player in self._players)

roster = Roster([
    Player("Ada", "Lovelace", 10, "MID"),
    Player("Grace", "Hopper", 7, "DEF"),
    Player("Katherine", "Johnson", 11, "MID"),
    Player("Margaret", "Hamilton", 3, "GK"),
])

print([player.last for player in roster])
print(list(roster.names()))
print(list(roster.numbers()))
print([player.first for player in roster.by_position("mid")])
print([player.last for player in roster.sorted_by_last_name()])
print(roster.position_counts())

assert len(roster) == 4
assert list(roster.names()) == [
    "Ada Lovelace",
    "Grace Hopper",
    "Katherine Johnson",
    "Margaret Hamilton",
]
assert list(roster.numbers()) == [10, 7, 11, 3]
assert [p.first for p in roster.by_position("MID")] == ["Ada", "Katherine"]
assert [p.last for p in roster.sorted_by_last_name()] == [
    "Hamilton",
    "Hopper",
    "Johnson",
    "Lovelace",
]


['Lovelace', 'Hopper', 'Johnson', 'Hamilton']
['Ada Lovelace', 'Grace Hopper', 'Katherine Johnson', 'Margaret Hamilton']
[10, 7, 11, 3]
['Ada', 'Katherine']
['Hamilton', 'Hopper', 'Johnson', 'Lovelace']
Counter({'MID': 2, 'DEF': 1, 'GK': 1})


### Best-practice note

A class can have one default iteration meaning, plus additional methods that expose alternate iteration views.

Good names for alternate views include:

- `items()`
- `keys()`
- `values()`
- `names()`
- `records()`
- `filtered(...)`
- `sorted_by_...()`


---

# Problem 3: Avoid the Iterator Exhaustion Trap

A common mistake is storing an iterator directly and then delegating to it.

That usually makes the object usable only once.

Build two versions:

1. `BadBatch`, which demonstrates the bug.
2. `GoodBatch`, which stores a reusable tuple and delegates correctly.
3. `FactoryBatch`, which accepts a factory function and can lazily create a new iterator for each pass.


In [5]:
# Problem 3 solution

class BadBatch:
    def __init__(self, records: Iterable[str]):
        # BUG: this stores a one-time iterator.
        self._records = iter(records)

    def __iter__(self) -> Iterator[str]:
        # Reusing the same iterator means the second pass will be empty.
        return self._records

class GoodBatch:
    def __init__(self, records: Iterable[str]):
        # Store a reusable iterable container.
        self._records = tuple(records)

    def __iter__(self) -> Iterator[str]:
        return iter(self._records)

class FactoryBatch:
    def __init__(self, record_factory: Callable[[], Iterable[str]]):
        self._record_factory = record_factory

    def __iter__(self) -> Iterator[str]:
        # Every call creates a fresh iterable and delegates to it.
        return iter(self._record_factory())

bad = BadBatch(["alpha", "beta", "gamma"])
print("Bad first pass:", list(bad))
print("Bad second pass:", list(bad))

assert list(bad) == []

good = GoodBatch(["alpha", "beta", "gamma"])
print("Good first pass:", list(good))
print("Good second pass:", list(good))

assert list(good) == ["alpha", "beta", "gamma"]
assert list(good) == ["alpha", "beta", "gamma"]

factory_batch = FactoryBatch(lambda: (str(n) for n in range(3)))
print("Factory first pass:", list(factory_batch))
print("Factory second pass:", list(factory_batch))

assert list(factory_batch) == ["0", "1", "2"]
assert list(factory_batch) == ["0", "1", "2"]


Bad first pass: ['alpha', 'beta', 'gamma']
Bad second pass: []
Good first pass: ['alpha', 'beta', 'gamma']
Good second pass: ['alpha', 'beta', 'gamma']
Factory first pass: ['0', '1', '2']
Factory second pass: ['0', '1', '2']


### Rule of thumb

If your class represents a collection, `__iter__` should usually return a **new** iterator each time.

Storing this is dangerous:

```python
self._items = iter(items)
```

Storing one of these is usually safer:

```python
self._items = tuple(items)
self._items = list(items)
self._factory = callable_that_returns_a_fresh_iterable
```


---

# Problem 4: Lazy Cleaning Pipeline

Create a `CleanLines` class.

Requirements:

1. Accept a source factory: a zero-argument function that returns an iterable of text lines.
2. On each iteration, produce cleaned lines.
3. Cleaning rules:
   - strip whitespace
   - ignore blank lines
   - ignore lines starting with `#`
   - collapse internal whitespace to a single space
4. Iteration must be lazy.
5. The object must be repeatable.


In [6]:
# Problem 4 solution

class CleanLines:
    def __init__(self, source_factory: Callable[[], Iterable[str]]):
        self._source_factory = source_factory

    @staticmethod
    def _clean(line: str) -> str:
        return " ".join(line.strip().split())

    def __iter__(self) -> Iterator[str]:
        # This generator is created fresh on every call to __iter__.
        for raw_line in self._source_factory():
            cleaned = self._clean(raw_line)
            if not cleaned:
                continue
            if cleaned.startswith("#"):
                continue
            yield cleaned

raw_lines = [
    "  first    line  ",
    "",
    "   # comment here",
    "second     line",
    " third\tline ",
]

clean_lines = CleanLines(lambda: raw_lines)

print(list(clean_lines))
print(list(clean_lines))

assert list(clean_lines) == ["first line", "second line", "third line"]
assert list(clean_lines) == ["first line", "second line", "third line"]


['first line', 'second line', 'third line']
['first line', 'second line', 'third line']


### Best-practice note

This design is useful for files, network pages, database cursors, or generated data.

For file data, you could use:

```python
CleanLines(lambda: open("data.txt", encoding="utf-8"))
```

Each iteration opens a fresh file handle, so repeated iteration is possible.


---

# Problem 5: Dictionary Delegation With Inventory

Create an `Inventory` class.

Requirements:

1. Store product quantities in an internal dictionary.
2. Iterating over `Inventory` should yield product names, like a dictionary does.
3. Add `items()` that delegates to the dictionary item view.
4. Add `values()` that delegates to the dictionary value view.
5. Add `in_stock()` that lazily yields product names whose quantity is positive.
6. Add `restock(product, amount)`.
7. Add tests for keys, items, values, and mutation.


In [7]:
# Problem 5 solution

class Inventory:
    def __init__(self, quantities: dict[str, int] | None = None):
        self._quantities = dict(quantities or {})

    def __iter__(self) -> Iterator[str]:
        # Delegates to dict key iteration.
        return iter(self._quantities)

    def __len__(self) -> int:
        return len(self._quantities)

    def __contains__(self, product: object) -> bool:
        return product in self._quantities

    def __getitem__(self, product: str) -> int:
        return self._quantities[product]

    def items(self):
        # Dictionary views are iterable and reflect later mutations.
        return self._quantities.items()

    def values(self):
        return self._quantities.values()

    def in_stock(self) -> Iterator[str]:
        return (
            product
            for product, quantity in self._quantities.items()
            if quantity > 0
        )

    def restock(self, product: str, amount: int) -> None:
        if amount < 0:
            raise ValueError("restock amount must be non-negative")
        self._quantities[product] = self._quantities.get(product, 0) + amount

    def sell(self, product: str, amount: int = 1) -> None:
        if amount < 0:
            raise ValueError("sell amount must be non-negative")
        available = self._quantities.get(product, 0)
        if amount > available:
            raise ValueError(f"not enough stock for {product!r}")
        self._quantities[product] = available - amount

inventory = Inventory({"notebook": 10, "pen": 0, "eraser": 5})

print(list(inventory))
print(list(inventory.items()))
print(list(inventory.values()))
print(list(inventory.in_stock()))

inventory.restock("pen", 3)
inventory.sell("notebook", 2)

print(list(inventory.items()))

assert list(inventory) == ["notebook", "pen", "eraser"]
assert list(inventory.in_stock()) == ["notebook", "pen", "eraser"]
assert inventory["notebook"] == 8
assert inventory["pen"] == 3


['notebook', 'pen', 'eraser']
[('notebook', 10), ('pen', 0), ('eraser', 5)]
[10, 0, 5]
['notebook', 'eraser']
[('notebook', 8), ('pen', 3), ('eraser', 5)]


### Live views vs snapshots

Dictionary views such as `dict.items()` are live views. They reflect later changes to the dictionary.

Sometimes that is exactly what you want.

Sometimes a snapshot is safer:

```python
return iter(tuple(self._quantities.items()))
```


---

# Problem 6: Snapshot Iteration During Mutation

Create two event log classes:

1. `LiveEventLog`, where iteration delegates directly to the internal list.
2. `SnapshotEventLog`, where iteration delegates to a tuple snapshot.

Then mutate the log during iteration and observe the difference.


In [8]:
# Problem 6 solution

class LiveEventLog:
    def __init__(self, events: Iterable[str] = ()): 
        self._events = list(events)

    def append(self, event: str) -> None:
        self._events.append(event)

    def __iter__(self) -> Iterator[str]:
        # Live list iterator can observe appended items during iteration.
        return iter(self._events)

class SnapshotEventLog:
    def __init__(self, events: Iterable[str] = ()): 
        self._events = list(events)

    def append(self, event: str) -> None:
        self._events.append(event)

    def __iter__(self) -> Iterator[str]:
        # Snapshot protects the current loop from later mutation.
        return iter(tuple(self._events))

live_log = LiveEventLog(["start", "validate"])
seen_live = []

for event in live_log:
    seen_live.append(event)
    if event == "start":
        live_log.append("appended during live iteration")

snapshot_log = SnapshotEventLog(["start", "validate"])
seen_snapshot = []

for event in snapshot_log:
    seen_snapshot.append(event)
    if event == "start":
        snapshot_log.append("appended during snapshot iteration")

print("live:", seen_live)
print("snapshot:", seen_snapshot)
print("snapshot second pass:", list(snapshot_log))

assert seen_live == ["start", "validate", "appended during live iteration"]
assert seen_snapshot == ["start", "validate"]
assert list(snapshot_log) == [
    "start",
    "validate",
    "appended during snapshot iteration",
]


live: ['start', 'validate', 'appended during live iteration']
snapshot: ['start', 'validate']
snapshot second pass: ['start', 'validate', 'appended during snapshot iteration']


### Design decision

Use live iteration when callers should see current state.

Use snapshot iteration when consistency during the loop matters more than seeing immediate changes.


---

# Problem 7: Delegating to `itertools.chain` for Multiple Sources

Create a `TagIndex` class.

Requirements:

1. Accept many groups of tags.
2. Iterating over the object should yield every tag from every group.
3. Add `unique()` to lazily yield unique tags in first-seen order.
4. Add `counts()` to count all tags.
5. Add `top(n)` to return the most common tags.


In [9]:
# Problem 7 solution

class TagIndex:
    def __init__(self, groups: Iterable[Iterable[str]]):
        # Convert each group to a tuple to make the whole object repeatable.
        self._groups = tuple(tuple(group) for group in groups)

    def __iter__(self) -> Iterator[str]:
        # Delegate to chain, which delegates to each group iterator.
        return chain.from_iterable(self._groups)

    def unique(self) -> Iterator[str]:
        seen: set[str] = set()
        for tag in self:
            normalized = tag.strip().lower()
            if normalized not in seen:
                seen.add(normalized)
                yield normalized

    def counts(self) -> Counter[str]:
        return Counter(tag.strip().lower() for tag in self)

    def top(self, n: int) -> list[tuple[str, int]]:
        return self.counts().most_common(n)

tag_index = TagIndex([
    ["python", "iterators", "python"],
    ["delegation", "containers"],
    ["Python", "advanced", "iterators"],
])

print(list(tag_index))
print(list(tag_index.unique()))
print(tag_index.counts())
print(tag_index.top(2))

assert list(tag_index) == [
    "python",
    "iterators",
    "python",
    "delegation",
    "containers",
    "Python",
    "advanced",
    "iterators",
]
assert list(tag_index.unique()) == [
    "python",
    "iterators",
    "delegation",
    "containers",
    "advanced",
]
assert tag_index.top(2) == [("python", 3), ("iterators", 2)]


['python', 'iterators', 'python', 'delegation', 'containers', 'Python', 'advanced', 'iterators']
['python', 'iterators', 'delegation', 'containers', 'advanced']
Counter({'python': 3, 'iterators': 2, 'delegation': 1, 'containers': 1, 'advanced': 1})
[('python', 3), ('iterators', 2)]


### Why convert groups to tuples?

If a group is a generator, it can be consumed once.

Converting each group to a tuple makes the whole `TagIndex` repeatable.

That is an eager design. For massive data, you might instead store factories that recreate each source lazily.


---

# Problem 8: Delegating Regex Matches Safely

`re.finditer(...)` returns an iterator. If you store that iterator, it gets exhausted.

Create a reusable `RegexMatches` class.

Requirements:

1. Accept a regex pattern and a text string.
2. Each iteration should produce `MatchInfo` objects.
3. A `MatchInfo` should contain `value`, `start`, and `end`.
4. Repeated iteration must work.
5. Add `values()` to iterate only over matched strings.


In [10]:
# Problem 8 solution

@dataclass(frozen=True)
class MatchInfo:
    value: str
    start: int
    end: int

class RegexMatches:
    def __init__(self, pattern: str, text: str, flags: int = 0):
        self._pattern = re.compile(pattern, flags)
        self._text = text

    def __iter__(self) -> Iterator[MatchInfo]:
        # Create a fresh re.finditer iterator each time.
        for match in self._pattern.finditer(self._text):
            yield MatchInfo(match.group(), match.start(), match.end())

    def values(self) -> Iterator[str]:
        return (match.value for match in self)

    def spans(self) -> Iterator[tuple[int, int]]:
        return ((match.start, match.end) for match in self)

text = "Order A-100, order B-205, and order C-999 are ready."
matches = RegexMatches(r"[A-Z]-\d+", text)

print(list(matches))
print(list(matches.values()))
print(list(matches.spans()))

assert list(matches.values()) == ["A-100", "B-205", "C-999"]
assert list(matches.values()) == ["A-100", "B-205", "C-999"]
assert list(matches.spans()) == [(6, 11), (19, 24), (36, 41)]


[MatchInfo(value='A-100', start=6, end=11), MatchInfo(value='B-205', start=19, end=24), MatchInfo(value='C-999', start=36, end=41)]
['A-100', 'B-205', 'C-999']
[(6, 11), (19, 24), (36, 41)]


### Best-practice note

Do not store `re.finditer(...)` directly when you need repeatable iteration.

Store the pattern and text, then call `finditer` inside `__iter__`.


---

# Problem 9: Delegating a Transaction Ledger

Create a `TransactionLedger` class.

Requirements:

1. Store immutable `Transaction` records.
2. Iterating over the ledger should yield transactions in insertion order.
3. Add `deposits()` and `withdrawals()` views.
4. Add `balance()`.
5. Add `largest_withdrawals(n)`.
6. Add `between(min_amount, max_amount)`.
7. Make all iteration repeatable.


In [11]:
# Problem 9 solution

@dataclass(frozen=True)
class Transaction:
    id: str
    description: str
    amount: float

class TransactionLedger:
    def __init__(self, transactions: Iterable[Transaction]):
        self._transactions = tuple(transactions)

    def __iter__(self) -> Iterator[Transaction]:
        return iter(self._transactions)

    def __len__(self) -> int:
        return len(self._transactions)

    def deposits(self) -> Iterator[Transaction]:
        return (transaction for transaction in self if transaction.amount > 0)

    def withdrawals(self) -> Iterator[Transaction]:
        return (transaction for transaction in self if transaction.amount < 0)

    def balance(self) -> float:
        return sum(transaction.amount for transaction in self)

    def largest_withdrawals(self, n: int) -> Iterator[Transaction]:
        withdrawals_by_size = sorted(
            self.withdrawals(),
            key=lambda transaction: abs(transaction.amount),
            reverse=True,
        )
        return iter(withdrawals_by_size[:n])

    def between(self, min_amount: float, max_amount: float) -> Iterator[Transaction]:
        return (
            transaction
            for transaction in self
            if min_amount <= transaction.amount <= max_amount
        )

ledger = TransactionLedger([
    Transaction("T001", "salary", 2500.00),
    Transaction("T002", "rent", -900.00),
    Transaction("T003", "groceries", -125.40),
    Transaction("T004", "book refund", 18.99),
    Transaction("T005", "utilities", -210.75),
])

print(list(ledger))
print(list(ledger.deposits()))
print(list(ledger.withdrawals()))
print(ledger.balance())
print(list(ledger.largest_withdrawals(2)))
print(list(ledger.between(-250, 20)))

assert len(ledger) == 5
assert [t.id for t in ledger.deposits()] == ["T001", "T004"]
assert [t.id for t in ledger.withdrawals()] == ["T002", "T003", "T005"]
assert round(ledger.balance(), 2) == 1282.84
assert [t.id for t in ledger.largest_withdrawals(2)] == ["T002", "T005"]
assert [t.id for t in ledger.between(-250, 20)] == ["T003", "T004", "T005"]


[Transaction(id='T001', description='salary', amount=2500.0), Transaction(id='T002', description='rent', amount=-900.0), Transaction(id='T003', description='groceries', amount=-125.4), Transaction(id='T004', description='book refund', amount=18.99), Transaction(id='T005', description='utilities', amount=-210.75)]
[Transaction(id='T001', description='salary', amount=2500.0), Transaction(id='T004', description='book refund', amount=18.99)]
[Transaction(id='T002', description='rent', amount=-900.0), Transaction(id='T003', description='groceries', amount=-125.4), Transaction(id='T005', description='utilities', amount=-210.75)]
1282.84
[Transaction(id='T002', description='rent', amount=-900.0), Transaction(id='T005', description='utilities', amount=-210.75)]
[Transaction(id='T003', description='groceries', amount=-125.4), Transaction(id='T004', description='book refund', amount=18.99), Transaction(id='T005', description='utilities', amount=-210.75)]


### Testing tip

Test custom iterables by consuming them in different ways:

```python
list(obj)
tuple(obj)
sorted(obj, key=...)
[x for x in obj]
next(iter(obj))
zip(obj, obj)
```

Repeated calls should behave consistently unless the object is intentionally one-shot.


---

# Problem 10: Paginated Data Delegation

Simulate a paginated data source.

Create a `PaginatedRecords` class.

Requirements:

1. Accept a `page_fetcher(page_number)` function.
2. The fetcher returns a list of records for that page.
3. An empty page means there are no more records.
4. Iterating over `PaginatedRecords` should lazily yield all records.
5. Repeated iteration should start from page 1 again.
6. Add `take(n)` that returns the first `n` records as a list.


In [12]:
# Problem 10 solution

class PaginatedRecords:
    def __init__(self, page_fetcher: Callable[[int], list[str]]):
        self._page_fetcher = page_fetcher

    def __iter__(self) -> Iterator[str]:
        page_number = 1
        while True:
            page = self._page_fetcher(page_number)
            if not page:
                break
            yield from page
            page_number += 1

    def take(self, n: int) -> list[str]:
        if n < 0:
            raise ValueError("n must be non-negative")
        return list(islice(self, n))

pages = {
    1: ["A", "B", "C"],
    2: ["D", "E"],
    3: ["F"],
    4: [],
}

fetch_calls: list[int] = []

def fetch_page(page_number: int) -> list[str]:
    fetch_calls.append(page_number)
    return pages.get(page_number, [])

records = PaginatedRecords(fetch_page)

print(records.take(4))
print(list(records))
print(list(records))
print(fetch_calls)

assert records.take(4) == ["A", "B", "C", "D"]
assert list(records) == ["A", "B", "C", "D", "E", "F"]
assert list(records) == ["A", "B", "C", "D", "E", "F"]


['A', 'B', 'C', 'D']
['A', 'B', 'C', 'D', 'E', 'F']
['A', 'B', 'C', 'D', 'E', 'F']
[1, 2, 1, 2, 3, 4, 1, 2, 3, 4]


### Best-practice note

This pattern is common when iterating over API pages, database pages, or search results.

`__iter__` does not store the page number on `self`, because that would make independent iterations interfere with each other.

The loop state belongs inside the iterator/generator created by `__iter__`.


---

# Problem 11: Custom Delegating Collection With Transform and Filter

Create a reusable `DelegatingCollection` class.

Requirements:

1. Accept any iterable.
2. Store data in a tuple.
3. Delegate iteration to the tuple.
4. Add `map(function)` that returns a new `DelegatingCollection`.
5. Add `filter(predicate)` that returns a new `DelegatingCollection`.
6. Add `sorted(key=None, reverse=False)` that returns a new `DelegatingCollection`.
7. Add `take(n)` that returns a tuple of the first `n` items.
8. Add tests that chain operations.


In [13]:
# Problem 11 solution

class DelegatingCollection:
    def __init__(self, items: Iterable):
        self._items = tuple(items)

    def __iter__(self) -> Iterator:
        return iter(self._items)

    def __len__(self) -> int:
        return len(self._items)

    def __contains__(self, item: object) -> bool:
        return item in self._items

    def __repr__(self) -> str:
        return f"DelegatingCollection({self._items!r})"

    def map(self, function: Callable) -> "DelegatingCollection":
        return DelegatingCollection(function(item) for item in self)

    def filter(self, predicate: Callable) -> "DelegatingCollection":
        return DelegatingCollection(item for item in self if predicate(item))

    def sorted(self, *, key: Callable | None = None, reverse: bool = False) -> "DelegatingCollection":
        return DelegatingCollection(sorted(self, key=key, reverse=reverse))

    def take(self, n: int) -> tuple:
        if n < 0:
            raise ValueError("n must be non-negative")
        return tuple(islice(self, n))

numbers = DelegatingCollection([5, 1, 9, 2, 7, 3])

result = (
    numbers
    .filter(lambda n: n % 2 == 1)
    .map(lambda n: n * n)
    .sorted(reverse=True)
    .take(3)
)

print(numbers)
print(result)

assert list(numbers) == [5, 1, 9, 2, 7, 3]
assert result == (81, 49, 25)
assert len(numbers) == 6
assert 9 in numbers
assert 4 not in numbers


DelegatingCollection((5, 1, 9, 2, 7, 3))
(81, 49, 25)


### Eager vs lazy design

The `DelegatingCollection` above is eager because every transformation builds a new tuple.

That is simple, repeatable, and safe.

For very large datasets, a lazy design might be better, but then you must be careful with one-shot iterators.


---

# Problem 12: Lazy Query Object With Repeatable Factories

Create a `LazyQuery` class that supports lazy chained operations while staying repeatable.

Requirements:

1. Accept a zero-argument factory that returns an iterable.
2. `__iter__` delegates to a fresh iterable from the factory.
3. Add `where(predicate)`.
4. Add `select(transform)`.
5. Add `take(n)`.
6. Chained operations should be lazy and repeatable.


In [14]:
# Problem 12 solution

class LazyQuery:
    def __init__(self, factory: Callable[[], Iterable]):
        self._factory = factory

    @classmethod
    def from_iterable(cls, iterable: Iterable) -> "LazyQuery":
        # Tuple snapshot makes even one-shot input repeatable after construction.
        items = tuple(iterable)
        return cls(lambda: items)

    def __iter__(self) -> Iterator:
        return iter(self._factory())

    def where(self, predicate: Callable) -> "LazyQuery":
        return LazyQuery(lambda: (item for item in self if predicate(item)))

    def select(self, transform: Callable) -> "LazyQuery":
        return LazyQuery(lambda: (transform(item) for item in self))

    def take(self, n: int) -> list:
        if n < 0:
            raise ValueError("n must be non-negative")
        return list(islice(self, n))

    def to_list(self) -> list:
        return list(self)

query = (
    LazyQuery.from_iterable(range(20))
    .where(lambda n: n % 3 == 0)
    .select(lambda n: {"number": n, "square": n * n})
)

print(query.take(4))
print(query.to_list())
print(query.to_list())

assert query.take(4) == [
    {"number": 0, "square": 0},
    {"number": 3, "square": 9},
    {"number": 6, "square": 36},
    {"number": 9, "square": 81},
]
assert len(query.to_list()) == 7
assert query.to_list() == query.to_list()


[{'number': 0, 'square': 0}, {'number': 3, 'square': 9}, {'number': 6, 'square': 36}, {'number': 9, 'square': 81}]
[{'number': 0, 'square': 0}, {'number': 3, 'square': 9}, {'number': 6, 'square': 36}, {'number': 9, 'square': 81}, {'number': 12, 'square': 144}, {'number': 15, 'square': 225}, {'number': 18, 'square': 324}]
[{'number': 0, 'square': 0}, {'number': 3, 'square': 9}, {'number': 6, 'square': 36}, {'number': 9, 'square': 81}, {'number': 12, 'square': 144}, {'number': 15, 'square': 225}, {'number': 18, 'square': 324}]


### Key idea

`LazyQuery` is repeatable because every `__iter__` call asks the factory for a fresh iterable.

Each chained query stores a new factory, not a consumed iterator.


---

# Problem 13: Delegating Nested Trees With Recursive Generators

Create a `TreeNode` class.

Requirements:

1. A node has a `value` and child nodes.
2. Iterating over a node should yield values in depth-first pre-order.
3. Add `children()` to iterate directly over child nodes.
4. Add `leaves()` to iterate over leaf values.
5. Add `find(predicate)` to return the first matching value or `None`.
6. Repeated iteration must work.


In [15]:
# Problem 13 solution

class TreeNode:
    def __init__(self, value, children: Iterable["TreeNode"] = ()): 
        self.value = value
        self._children = tuple(children)

    def __iter__(self) -> Iterator:
        # First yield this node's value.
        yield self.value

        # Then delegate to each child node's iterator.
        for child in self._children:
            yield from child

    def children(self) -> Iterator["TreeNode"]:
        return iter(self._children)

    def leaves(self) -> Iterator:
        if not self._children:
            yield self.value
            return
        for child in self._children:
            yield from child.leaves()

    def find(self, predicate: Callable) -> object | None:
        for value in self:
            if predicate(value):
                return value
        return None

root = TreeNode(
    "A",
    [
        TreeNode("B", [TreeNode("D"), TreeNode("E")]),
        TreeNode("C", [TreeNode("F")]),
    ],
)

print(list(root))
print([child.value for child in root.children()])
print(list(root.leaves()))
print(root.find(lambda value: value == "E"))
print(root.find(lambda value: value == "Z"))

assert list(root) == ["A", "B", "D", "E", "C", "F"]
assert list(root.leaves()) == ["D", "E", "F"]
assert root.find(lambda value: value == "E") == "E"
assert root.find(lambda value: value == "Z") is None
assert list(root) == list(root)


['A', 'B', 'D', 'E', 'C', 'F']
['B', 'C']
['D', 'E', 'F']
E
None


### Delegation syntax

`yield from child` means:

```python
for item in child:
    yield item
```

It delegates part of the iteration to another iterable.


---

# Problem 14: Ordered Lookup With Delegated Iteration

Create an `OrderedLookup` class.

Requirements:

1. Preserve insertion order.
2. Iterating over the object yields keys.
3. `items()` yields key-value pairs.
4. `values()` yields values.
5. `reversed_keys()` yields keys in reverse insertion order.
6. Updating an existing key should not change its original position.


In [16]:
# Problem 14 solution

class OrderedLookup:
    def __init__(self, pairs: Iterable[tuple[str, object]] = ()): 
        self._data = OrderedDict(pairs)

    def __iter__(self) -> Iterator[str]:
        return iter(self._data)

    def __len__(self) -> int:
        return len(self._data)

    def __setitem__(self, key: str, value: object) -> None:
        self._data[key] = value

    def __getitem__(self, key: str) -> object:
        return self._data[key]

    def items(self):
        return self._data.items()

    def values(self):
        return self._data.values()

    def reversed_keys(self) -> Iterator[str]:
        return reversed(self._data)

lookup = OrderedLookup([
    ("first", 10),
    ("second", 20),
    ("third", 30),
])

lookup["second"] = 200
lookup["fourth"] = 40

print(list(lookup))
print(list(lookup.items()))
print(list(lookup.values()))
print(list(lookup.reversed_keys()))

assert list(lookup) == ["first", "second", "third", "fourth"]
assert list(lookup.items()) == [
    ("first", 10),
    ("second", 200),
    ("third", 30),
    ("fourth", 40),
]
assert list(lookup.reversed_keys()) == ["fourth", "third", "second", "first"]


['first', 'second', 'third', 'fourth']
[('first', 10), ('second', 200), ('third', 30), ('fourth', 40)]
[10, 200, 30, 40]
['fourth', 'third', 'second', 'first']


### Modern Python note

Regular dictionaries preserve insertion order in modern Python, but `OrderedDict` still communicates intent clearly and has extra order-related methods.


---

# Problem 15: Build a Read-Only Playlist

Create a `Playlist` class.

Requirements:

1. Store immutable `Song` objects.
2. Iterating over a playlist should yield songs.
3. Add `titles()`.
4. Add `artists()`.
5. Add `total_seconds()`.
6. Add `longer_than(seconds)`.
7. Add `sorted_by_duration()`.
8. Do not expose the internal list or tuple directly as mutable state.


In [17]:
# Problem 15 solution

@dataclass(frozen=True)
class Song:
    title: str
    artist: str
    seconds: int

class Playlist:
    def __init__(self, songs: Iterable[Song]):
        self._songs = tuple(songs)

    def __iter__(self) -> Iterator[Song]:
        return iter(self._songs)

    def __len__(self) -> int:
        return len(self._songs)

    def titles(self) -> Iterator[str]:
        return (song.title for song in self)

    def artists(self) -> Iterator[str]:
        return (song.artist for song in self)

    def total_seconds(self) -> int:
        return sum(song.seconds for song in self)

    def longer_than(self, seconds: int) -> Iterator[Song]:
        return (song for song in self if song.seconds > seconds)

    def sorted_by_duration(self, *, reverse: bool = False) -> Iterator[Song]:
        return iter(sorted(self, key=lambda song: song.seconds, reverse=reverse))

    def as_tuple(self) -> tuple[Song, ...]:
        return self._songs

playlist = Playlist([
    Song("Intro", "The Iterables", 95),
    Song("Delegate", "The Iterables", 210),
    Song("Fresh Iterator", "Loop Theory", 185),
    Song("Yield From", "Generator Band", 240),
])

print(list(playlist.titles()))
print(list(playlist.artists()))
print(playlist.total_seconds())
print([song.title for song in playlist.longer_than(200)])
print([song.title for song in playlist.sorted_by_duration(reverse=True)])

assert list(playlist.titles()) == ["Intro", "Delegate", "Fresh Iterator", "Yield From"]
assert playlist.total_seconds() == 730
assert [song.title for song in playlist.longer_than(200)] == ["Delegate", "Yield From"]
assert [song.title for song in playlist.sorted_by_duration()] == [
    "Intro",
    "Fresh Iterator",
    "Delegate",
    "Yield From",
]


['Intro', 'Delegate', 'Fresh Iterator', 'Yield From']
['The Iterables', 'The Iterables', 'Loop Theory', 'Generator Band']
730
['Delegate', 'Yield From']
['Yield From', 'Delegate', 'Fresh Iterator', 'Intro']


---

# Problem 16: Composite Report With Section Delegation

Create a `Report` class made of `Section` objects.

Requirements:

1. A `Section` has a title and lines.
2. Iterating over a `Section` yields its lines.
3. Iterating over a `Report` yields all lines from all sections.
4. Add `section_titles()`.
5. Add `numbered_lines()` yielding `(line_number, line)`.
6. Add `search(term)` yielding matching lines.
7. Use `yield from` for nested delegation.


In [18]:
# Problem 16 solution

@dataclass(frozen=True)
class Section:
    title: str
    lines: tuple[str, ...]

    def __iter__(self) -> Iterator[str]:
        return iter(self.lines)

class Report:
    def __init__(self, sections: Iterable[Section]):
        self._sections = tuple(sections)

    def __iter__(self) -> Iterator[str]:
        for section in self._sections:
            yield from section

    def sections(self) -> Iterator[Section]:
        return iter(self._sections)

    def section_titles(self) -> Iterator[str]:
        return (section.title for section in self._sections)

    def numbered_lines(self, start: int = 1) -> Iterator[tuple[int, str]]:
        for line_number, line in enumerate(self, start=start):
            yield line_number, line

    def search(self, term: str) -> Iterator[str]:
        normalized = term.casefold()
        return (line for line in self if normalized in line.casefold())

report = Report([
    Section("Overview", ("Delegation reduces boilerplate.", "Iterables are reusable when designed well.")),
    Section("Warnings", ("Do not store one-shot iterators.", "Return fresh iterators from __iter__.")),
])

print(list(report.section_titles()))
print(list(report))
print(list(report.numbered_lines()))
print(list(report.search("iterator")))

assert list(report.section_titles()) == ["Overview", "Warnings"]
assert list(report) == [
    "Delegation reduces boilerplate.",
    "Iterables are reusable when designed well.",
    "Do not store one-shot iterators.",
    "Return fresh iterators from __iter__.",
]
assert list(report.numbered_lines(start=10))[0] == (10, "Delegation reduces boilerplate.")
assert list(report.search("iterator")) == [
    "Do not store one-shot iterators.",
    "Return fresh iterators from __iter__.",
]


['Overview', 'Warnings']
['Delegation reduces boilerplate.', 'Iterables are reusable when designed well.', 'Do not store one-shot iterators.', 'Return fresh iterators from __iter__.']
[(1, 'Delegation reduces boilerplate.'), (2, 'Iterables are reusable when designed well.'), (3, 'Do not store one-shot iterators.'), (4, 'Return fresh iterators from __iter__.')]
['Do not store one-shot iterators.', 'Return fresh iterators from __iter__.']


---

# Problem 17: Iterator Independence Test

Create a helper function `assert_independent_iterators(iterable)`.

It should verify that two iterators created from the same iterable do not accidentally share progress.

Use it on:

1. `NameDirectory`
2. `Roster`
3. `GoodBatch`
4. A deliberately bad iterable


In [19]:
# Problem 17 solution

def assert_independent_iterators(iterable, expected_first_two: tuple[object, object]) -> None:
    iterator_a = iter(iterable)
    iterator_b = iter(iterable)

    first_a = next(iterator_a)
    first_b = next(iterator_b)
    second_a = next(iterator_a)
    second_b = next(iterator_b)

    assert (first_a, second_a) == expected_first_two
    assert (first_b, second_b) == expected_first_two

assert_independent_iterators(directory, ("Graham Chapman", "John Cleese"))
assert_independent_iterators(roster, tuple(list(roster)[:2]))
assert_independent_iterators(good, ("alpha", "beta"))

bad_again = BadBatch(["x", "y", "z"])
iterator_a = iter(bad_again)
iterator_b = iter(bad_again)

print(next(iterator_a))
print(next(iterator_b))  # This is y, not x, because both names point to the same iterator.

assert next(iterator_a) == "z"


x
y


### What the helper proves

For a proper iterable container:

```python
iter_a = iter(container)
iter_b = iter(container)
```

`iter_a` and `iter_b` should usually progress independently.

For an iterator object:

```python
iter(iterator_obj) is iterator_obj
```

Progress is shared because the object is the iterator.


---

# Problem 18: Challenge — Student GradeBook

Create a complete `GradeBook` class.

Requirements:

1. Store `Student(name, grades)` objects.
2. Iterating over the grade book should yield students sorted by name.
3. Add `names()`.
4. Add `averages()` yielding `(name, average)` pairs.
5. Add `passing(min_average=60)` yielding students with average above the threshold.
6. Add `top(n)` yielding the top `n` students by average.
7. Add `grade_rows()` yielding dictionaries suitable for exporting.
8. Use delegated iteration and generator-based views.


In [20]:
# Problem 18 solution

@dataclass(frozen=True)
class Student:
    name: str
    grades: tuple[float, ...]

    @property
    def average(self) -> float:
        if not self.grades:
            return 0.0
        return sum(self.grades) / len(self.grades)

class GradeBook:
    def __init__(self, students: Iterable[Student]):
        self._students = tuple(students)

    def __iter__(self) -> Iterator[Student]:
        # Default view is alphabetical, not insertion order.
        return iter(sorted(self._students, key=lambda student: student.name.casefold()))

    def __len__(self) -> int:
        return len(self._students)

    def names(self) -> Iterator[str]:
        return (student.name for student in self)

    def averages(self) -> Iterator[tuple[str, float]]:
        return ((student.name, student.average) for student in self)

    def passing(self, min_average: float = 60) -> Iterator[Student]:
        return (student for student in self if student.average >= min_average)

    def top(self, n: int) -> Iterator[Student]:
        if n < 0:
            raise ValueError("n must be non-negative")
        ranked = sorted(self._students, key=lambda student: student.average, reverse=True)
        return iter(ranked[:n])

    def grade_rows(self) -> Iterator[dict[str, object]]:
        return (
            {
                "name": student.name,
                "grades": student.grades,
                "average": round(student.average, 2),
                "passing": student.average >= 60,
            }
            for student in self
        )

grade_book = GradeBook([
    Student("Zoe", (91, 88, 95)),
    Student("Ana", (72, 68, 70)),
    Student("Miro", (55, 58, 62)),
    Student("Boris", (100, 94, 98)),
])

print(list(grade_book.names()))
print(list(grade_book.averages()))
print([student.name for student in grade_book.passing(70)])
print([student.name for student in grade_book.top(2)])
print(list(grade_book.grade_rows()))

assert list(grade_book.names()) == ["Ana", "Boris", "Miro", "Zoe"]
assert [student.name for student in grade_book.passing(70)] == ["Ana", "Boris", "Zoe"]
assert [student.name for student in grade_book.top(2)] == ["Boris", "Zoe"]
assert list(grade_book.names()) == list(grade_book.names())


['Ana', 'Boris', 'Miro', 'Zoe']
[('Ana', 70.0), ('Boris', 97.33333333333333), ('Miro', 58.333333333333336), ('Zoe', 91.33333333333333)]
['Ana', 'Boris', 'Zoe']
['Boris', 'Zoe']
[{'name': 'Ana', 'grades': (72, 68, 70), 'average': 70.0, 'passing': True}, {'name': 'Boris', 'grades': (100, 94, 98), 'average': 97.33, 'passing': True}, {'name': 'Miro', 'grades': (55, 58, 62), 'average': 58.33, 'passing': False}, {'name': 'Zoe', 'grades': (91, 88, 95), 'average': 91.33, 'passing': True}]


---

# Problem 19: Challenge — Windowed Iterable

Create a `Windowed` class that delegates to generated windows over stored data.

Requirements:

1. Accept any iterable of items.
2. Accept a positive window size.
3. Iterating over `Windowed` should yield fixed-size tuples.
4. Add `partial=True` support so the final incomplete window can be included.
5. Repeated iteration must work.

Example:

```python
Windowed([1, 2, 3, 4, 5], size=2)
# yields (1, 2), (3, 4)

Windowed([1, 2, 3, 4, 5], size=2, partial=True)
# yields (1, 2), (3, 4), (5,)
```


In [21]:
# Problem 19 solution

class Windowed:
    def __init__(self, items: Iterable, size: int, *, partial: bool = False):
        if size <= 0:
            raise ValueError("size must be positive")
        self._items = tuple(items)
        self._size = size
        self._partial = partial

    def __iter__(self) -> Iterator[tuple]:
        for index in range(0, len(self._items), self._size):
            window = self._items[index:index + self._size]
            if len(window) == self._size or self._partial:
                yield window

    def flattened(self) -> Iterator:
        return chain.from_iterable(self)

windows = Windowed([1, 2, 3, 4, 5], 2)
partial_windows = Windowed([1, 2, 3, 4, 5], 2, partial=True)

print(list(windows))
print(list(partial_windows))
print(list(partial_windows.flattened()))

assert list(windows) == [(1, 2), (3, 4)]
assert list(partial_windows) == [(1, 2), (3, 4), (5,)]
assert list(partial_windows.flattened()) == [1, 2, 3, 4, 5]
assert list(partial_windows) == list(partial_windows)


[(1, 2), (3, 4)]
[(1, 2), (3, 4), (5,)]
[1, 2, 3, 4, 5]


---

# Problem 20: Final Capstone — Mini DataFrame Rows

Create a small `Rows` class that wraps a sequence of dictionaries.

Requirements:

1. Iterating over `Rows` yields row dictionaries.
2. Store copies of rows to avoid accidental external mutation.
3. Add `select(*columns)` yielding smaller dictionaries.
4. Add `where(predicate)` yielding matching rows.
5. Add `order_by(column, reverse=False)` returning a new `Rows` object.
6. Add `column(column_name)` yielding values from one column.
7. Add `to_list()` returning a list of row copies.
8. Demonstrate chained usage.


In [22]:
# Problem 20 solution

class Rows:
    def __init__(self, rows: Iterable[dict[str, object]]):
        self._rows = tuple(dict(row) for row in rows)

    def __iter__(self) -> Iterator[dict[str, object]]:
        # Yield copies so callers cannot mutate internal state.
        return (dict(row) for row in self._rows)

    def __len__(self) -> int:
        return len(self._rows)

    def select(self, *columns: str) -> Iterator[dict[str, object]]:
        return (
            {column: row[column] for column in columns}
            for row in self
        )

    def where(self, predicate: Callable[[dict[str, object]], bool]) -> Iterator[dict[str, object]]:
        return (row for row in self if predicate(row))

    def order_by(self, column: str, *, reverse: bool = False) -> "Rows":
        return Rows(sorted(self, key=lambda row: row[column], reverse=reverse))

    def column(self, column_name: str) -> Iterator[object]:
        return (row[column_name] for row in self)

    def to_list(self) -> list[dict[str, object]]:
        return list(self)

rows = Rows([
    {"name": "Ada", "team": "A", "score": 91},
    {"name": "Grace", "team": "B", "score": 88},
    {"name": "Katherine", "team": "A", "score": 95},
    {"name": "Margaret", "team": "B", "score": 99},
])

team_a_sorted = Rows(rows.where(lambda row: row["team"] == "A")).order_by("score", reverse=True)

print(rows.to_list())
print(list(rows.column("name")))
print(list(rows.select("name", "score")))
print(team_a_sorted.to_list())

# Prove that returned rows are defensive copies.
external = rows.to_list()
external[0]["score"] = -1

assert list(rows.column("score")) == [91, 88, 95, 99]
assert list(rows.column("name")) == ["Ada", "Grace", "Katherine", "Margaret"]
assert team_a_sorted.to_list() == [
    {"name": "Katherine", "team": "A", "score": 95},
    {"name": "Ada", "team": "A", "score": 91},
]
assert len(rows) == 4


[{'name': 'Ada', 'team': 'A', 'score': 91}, {'name': 'Grace', 'team': 'B', 'score': 88}, {'name': 'Katherine', 'team': 'A', 'score': 95}, {'name': 'Margaret', 'team': 'B', 'score': 99}]
['Ada', 'Grace', 'Katherine', 'Margaret']
[{'name': 'Ada', 'score': 91}, {'name': 'Grace', 'score': 88}, {'name': 'Katherine', 'score': 95}, {'name': 'Margaret', 'score': 99}]
[{'name': 'Katherine', 'team': 'A', 'score': 95}, {'name': 'Ada', 'team': 'A', 'score': 91}]


---

# Final Best Practices Checklist

Use this checklist when designing iterable classes:

1. **Delegate when possible**: `return iter(self._items)` is often enough.
2. **Return a fresh iterator** from `__iter__` for collection-like objects.
3. **Do not store one-shot iterators** unless the class is intentionally one-shot.
4. **Use tuples for safe, repeatable, immutable internal storage**.
5. **Use generator methods for filtered or transformed views**.
6. **Use `yield from` for nested delegation**.
7. **Decide between live views and snapshots** deliberately.
8. **Expose behavior, not internals**.
9. **Test repeated iteration** with `list(obj)` twice.
10. **Test iterator independence** with two simultaneous iterators.


In [23]:
# Final quick self-test

all_checks = []

all_checks.append(list(directory) == [
    "Graham Chapman",
    "John Cleese",
    "Terry Gilliam",
    "Eric Idle",
])

all_checks.append([player.first for player in roster.by_position("MID")] == ["Ada", "Katherine"])
all_checks.append(list(good) == ["alpha", "beta", "gamma"])
all_checks.append(list(clean_lines) == ["first line", "second line", "third line"])
all_checks.append(list(inventory.in_stock()) == ["notebook", "pen", "eraser"])
all_checks.append(list(root.leaves()) == ["D", "E", "F"])
all_checks.append([student.name for student in grade_book.top(2)] == ["Boris", "Zoe"])
all_checks.append(list(partial_windows.flattened()) == [1, 2, 3, 4, 5])
all_checks.append(list(rows.column("score")) == [91, 88, 95, 99])

print(all_checks)
assert all(all_checks)
print("All delegating iterator examples passed.")


[True, True, True, True, True, True, True, True, True]
All delegating iterator examples passed.
